# 10 · Build and train a tiny detector from scratch
**Level:** advanced foundation · **Time:** 60–90 minutes including discussion · **Runs offline on CPU**

**You will:** generate independent datasets, build a convolutional network with two prediction heads, combine classification and localization losses, select a model using validation data, and evaluate held-out predictions.

**Prerequisites:** notebooks 01 and 03, familiarity with Python functions/classes, basic PyTorch tensors, and the project PyTorch installation.

**Scope:** every image contains **exactly one** rectangle, circle, or triangle. The network predicts one class and one box. It has no background/objectness output, cannot represent multiple objects, and is not a replacement for YOLO or Faster R-CNN. This controlled problem exposes the learning mechanics of a detector.

Default training uses small 48×48 images and a short CPU run; duration depends on your laptop. `COURSE_SMOKE=1` uses a very small run for code verification, not model-quality claims.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


In [ ]:
import os
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

seed_everything(42)
torch.manual_seed(42)
torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device("cpu")  # predictable, no GPU required
SMOKE = os.environ.get("COURSE_SMOKE", "0") == "1"
SIZE = 48
CLASS_NAMES = ["rectangle", "circle", "triangle"]
TRAIN_N, VAL_N, TEST_N = (96, 30, 30) if SMOKE else (1200, 240, 240)
EPOCHS = 2 if SMOKE else 18
print("PyTorch:", torch.__version__, "| device:", DEVICE, "| epochs:", EPOCHS)


## Step 1 — Generate one labeled shape per image
Each box is normalized to `[0, 1]`. Shape color is sampled independently of class, so the model cannot solve classification by a fixed class-to-color lookup. Position, dimensions, color, and background vary.

Train, validation, and test use different random seeds. Validation is used to select a checkpoint. Test stays out of training and checkpoint selection. Synthetic held-out results describe this generator's distribution, not photographs or real-world performance.


In [ ]:
def build_dataset(count, seed, shift=False):
    rng = np.random.default_rng(seed)
    images, classes, boxes = [], [], []
    for index in range(count):
        class_id = index % 3  # balanced classes; batches are shuffled later
        w, h = rng.integers(14, 29, size=2)
        if class_id == 1:
            h = w  # equal axes keep the circle class geometrically a circle
        x1 = int(rng.integers(2, SIZE - w - 1))
        y1 = int(rng.integers(2, SIZE - h - 1))
        x2, y2 = x1 + int(w), y1 + int(h)
        background = int(rng.integers(5, 40) if not shift else rng.integers(170, 210))
        picture = Image.new("RGB", (SIZE, SIZE), (background,) * 3)
        painter = ImageDraw.Draw(picture)
        color = tuple(int(c) for c in rng.integers(100, 245, size=3))
        if class_id == 0:
            painter.rectangle((x1, y1, x2 - 1, y2 - 1), fill=color)
        elif class_id == 1:
            painter.ellipse((x1, y1, x2 - 1, y2 - 1), fill=color)
        else:
            painter.polygon([(x1, y2 - 1), (x2 - 1, y2 - 1),
                             ((x1 + x2 - 1) // 2, y1)], fill=color)
        array = np.asarray(picture).astype(np.float32)
        array = np.clip(array + rng.normal(0, 2, array.shape), 0, 255) / 255.0
        images.append(array.transpose(2, 0, 1))  # HWC -> CHW
        classes.append(class_id)
        boxes.append(np.array([x1, y1, x2, y2], dtype=np.float32) / SIZE)
    return TensorDataset(torch.tensor(np.stack(images), dtype=torch.float32),
                         torch.tensor(classes, dtype=torch.long),
                         torch.tensor(np.stack(boxes), dtype=torch.float32))

train_data = build_dataset(TRAIN_N, seed=100)
val_data = build_dataset(VAL_N, seed=200)
test_data = build_dataset(TEST_N, seed=300)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=0)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=0)
print("Independent samples:", len(train_data), "train /", len(val_data), "validation /", len(test_data), "test")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for index, ax in enumerate(axes.flat):
    image_tensor, class_id, box = train_data[index]
    show_boxes(image_tensor.permute(1, 2, 0).numpy(), [box.numpy() * SIZE],
               [CLASS_NAMES[class_id.item()]], ax=ax, title=f"Training sample {index}")
plt.tight_layout()
plt.show()


## Step 2 — Build a shared backbone and two heads
Convolutions learn local patterns. The flattened spatial feature map retains position information needed for localization. A shared feature vector feeds a three-class head and a four-number box head.

The class head returns **logits**, not probabilities; `CrossEntropyLoss` expects logits. The box head transforms its values into valid normalized corners: width/height are in `(0,1)` and the top-left corner is constrained so the whole box remains inside the image.


In [ ]:
class TinySingleObjectDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 12, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(12, 24, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(24, 24, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(), nn.Linear(24 * 12 * 12, 128), nn.ReLU())
        self.class_head = nn.Linear(128, 3)
        self.box_head = nn.Linear(128, 4)

    def forward(self, images):
        features = self.features(images)
        logits = self.class_head(features)
        raw = self.box_head(features).sigmoid()
        sizes = raw[:, 2:]
        top_left = raw[:, :2] * (1 - sizes)
        boxes = torch.cat([top_left, top_left + sizes], dim=1)
        return logits, boxes

model = TinySingleObjectDetector().to(DEVICE)
demo_logits, demo_boxes = model(train_data.tensors[0][:4].to(DEVICE))
print("Class logits:", tuple(demo_logits.shape), "| boxes:", tuple(demo_boxes.shape))
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))
assert torch.all(demo_boxes[:, 2:] > demo_boxes[:, :2])


## Step 3 — Define losses and useful metrics
We minimize `classification loss + 10 × box loss`. The weight balances different numerical scales; it is a tunable choice, not a universal constant. Smooth L1 penalizes coordinate errors. IoU assesses overlap.

We report class accuracy, mean IoU, and the fraction of images with **both correct class and IoU ≥ 0.5**. That last measure is a simple one-object success rate, **not mAP**. Multibox detector evaluation requires confidence ranking and matching predictions to multiple ground-truth objects.


In [ ]:
class_loss_fn = nn.CrossEntropyLoss()
box_loss_fn = nn.SmoothL1Loss(beta=0.1)
BOX_WEIGHT = 10.0

def aligned_iou(predicted, target):
    top_left = torch.maximum(predicted[:, :2], target[:, :2])
    bottom_right = torch.minimum(predicted[:, 2:], target[:, 2:])
    intersection = (bottom_right - top_left).clamp(min=0).prod(dim=1)
    pred_area = (predicted[:, 2:] - predicted[:, :2]).clamp(min=0).prod(dim=1)
    true_area = (target[:, 2:] - target[:, :2]).clamp(min=0).prod(dim=1)
    return intersection / (pred_area + true_area - intersection).clamp(min=1e-8)

@torch.no_grad()
def evaluate(current_model, loader):
    current_model.eval()
    total, loss_sum, correct_sum, iou_sum, success_sum = 0, 0.0, 0, 0.0, 0
    for images, labels, target_boxes in loader:
        images, labels, target_boxes = images.to(DEVICE), labels.to(DEVICE), target_boxes.to(DEVICE)
        logits, predicted_boxes = current_model(images)
        loss = class_loss_fn(logits, labels) + BOX_WEIGHT * box_loss_fn(predicted_boxes, target_boxes)
        ious = aligned_iou(predicted_boxes, target_boxes)
        correct = logits.argmax(dim=1) == labels
        total += len(images)
        loss_sum += loss.item() * len(images)
        correct_sum += correct.sum().item()
        iou_sum += ious.sum().item()
        success_sum += (correct & (ious >= 0.5)).sum().item()
    return {"loss": loss_sum / total, "class_accuracy": correct_sum / total,
            "mean_iou": iou_sum / total, "joint_success": success_sum / total}

print("Untrained validation baseline:", evaluate(model, val_loader))


## Step 4 — Train, validate, and retain the best validation checkpoint
Each batch follows: zero gradients → forward pass → loss → backpropagation → optimizer update. Validation uses evaluation mode and disables gradient tracking. Selecting by validation loss keeps the test set out of model selection.

**Presenter prompt:** pause after the first epoch. Ask whether lower training loss alone proves that the model generalizes. Inspect validation results before answering.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
best_loss, best_epoch, best_state = float("inf"), 0, None
start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    training_sum = 0.0
    for images, labels, target_boxes in train_loader:
        images, labels, target_boxes = images.to(DEVICE), labels.to(DEVICE), target_boxes.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, predicted_boxes = model(images)
        loss = class_loss_fn(logits, labels) + BOX_WEIGHT * box_loss_fn(predicted_boxes, target_boxes)
        loss.backward()
        optimizer.step()
        training_sum += loss.item() * len(images)
    validation = evaluate(model, val_loader)
    history.append({"epoch": epoch, "train_loss": training_sum / len(train_data), **validation})
    if validation["loss"] < best_loss:
        best_loss, best_epoch = validation["loss"], epoch
        best_state = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch:02d}: train={history[-1]['train_loss']:.3f} "
          f"val={validation['loss']:.3f} class_acc={validation['class_accuracy']:.3f} "
          f"IoU={validation['mean_iou']:.3f} joint={validation['joint_success']:.3f}")
model.load_state_dict(best_state)
print(f"Restored epoch {best_epoch}; training took {time.perf_counter() - start:.1f} seconds.")


In [ ]:
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [row["train_loss"] for row in history], label="Train")
axes[0].plot(epochs, [row["loss"] for row in history], label="Validation")
axes[0].set(xlabel="Epoch", ylabel="Combined loss", title="Learning curves")
axes[0].legend()
for metric in ["class_accuracy", "mean_iou", "joint_success"]:
    axes[1].plot(epochs, [row[metric] for row in history], label=metric)
axes[1].set(xlabel="Epoch", ylabel="Validation metric", ylim=(0, 1.05))
axes[1].legend()
plt.tight_layout()
plt.show()


**Expected observation:** a complete default run should usually improve over the untrained validation baseline. The precise curve depends on numerical libraries and training randomness. Poor results are evidence to investigate: inspect labels, images, losses, and validation curves. A two-epoch smoke run can remain inaccurate and still verify that the code executes.

## Step 5 — Evaluate once on the held-out test set
This is the performance estimate for the chosen checkpoint. If you repeatedly tune after seeing these results, this set becomes another validation set; reserve a fresh final test set before making a final claim.


In [ ]:
test_metrics = evaluate(model, test_loader)
print("Held-out test results:")
for name, value in test_metrics.items():
    print(f"  {name}: {value:.4f}")
print("These are synthetic one-object metrics, not real-world mAP.")


## Step 6 — Inspect examples, including localization failures
Draw green ground-truth boxes and red predicted boxes. A softmax score expresses relative preference among our three classes; this model always chooses one even when shown an empty or unrelated image. It is not an object-presence probability.


In [ ]:
from matplotlib.patches import Rectangle
model.eval()
with torch.no_grad():
    test_logits, test_boxes = model(test_data.tensors[0].to(DEVICE))
    probabilities = test_logits.softmax(dim=1).cpu()
    test_boxes = test_boxes.cpu()
    ious = aligned_iou(test_boxes, test_data.tensors[2])
worst = torch.argsort(ious)[:3].tolist()
examples = [0, 1, 2] + worst
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, index in zip(axes.flat, examples):
    picture, true_class, true_box = test_data[index]
    ax.imshow(picture.permute(1, 2, 0).numpy())
    for box, color in [(true_box, "lime"), (test_boxes[index], "red")]:
        x1, y1, x2, y2 = (box.numpy() * SIZE).tolist()
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2))
    score, predicted_class = probabilities[index].max(dim=0)
    ax.set_title(f"True: {CLASS_NAMES[true_class.item()]} | Pred: {CLASS_NAMES[predicted_class.item()]}\n"
                 f"score={score.item():.2f}, IoU={ious[index].item():.2f}")
    ax.axis("off")
plt.suptitle("Top: first examples · bottom: lowest IoU examples · green=true, red=prediction")
plt.tight_layout()
plt.show()


## Step 7 — Test a distribution change and save the trained weights
A bright background was absent from training. Keep the shapes and task the same, then change only this background distribution. Compare the metrics; do not assume performance transfers.

The saved file contains weights and small configuration values. It is written to `outputs/tiny_detector.pt`; rerunning this cell replaces the course checkpoint.


In [ ]:
shifted_data = build_dataset(TEST_N, seed=400, shift=True)
shifted_loader = DataLoader(shifted_data, batch_size=128, shuffle=False, num_workers=0)
shifted_metrics = evaluate(model, shifted_loader)
print("Original test:", test_metrics)
print("Bright-background shift:", shifted_metrics)
output_dir = ROOT / "outputs"
output_dir.mkdir(exist_ok=True)
checkpoint_path = output_dir / "tiny_detector.pt"
torch.save({"model_state": model.state_dict(), "class_names": CLASS_NAMES,
            "image_size": SIZE, "selected_epoch": best_epoch}, checkpoint_path)
print("Saved:", checkpoint_path)


## Try it yourself
1. Explain why changing `BOX_WEIGHT` changes the tradeoff between class and box learning. Compare validation results after restarting and retraining with 1.0 and 20.0; keep the test set untouched during this comparison.
2. Add bright backgrounds to the **training generator**, then train a fresh model. Use a separate shifted validation set to select settings.
3. Reload the checkpoint into a new model and verify identical predictions on the same inputs.
4. Explain what must change to detect zero, one, or many objects per image.


## Solutions and discussion
1. A larger box weight increases localization's contribution to the shared feature gradients. It does not guarantee better IoU; measure validation behavior.
2. Randomly choose a dark or bright background for each training sample. Preserve independent validation/test seeds. Simply evaluating on shifted data does not adapt the model.
3. The reload demonstration is below. Load only checkpoints you created or trust.
4. A general detector needs a variable number of candidate predictions, an object/background mechanism, target assignment, and appropriate loss/evaluation logic. A single four-coordinate head cannot encode several objects.


In [ ]:
saved = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
restored_model = TinySingleObjectDetector().to(DEVICE)
restored_model.load_state_dict(saved["model_state"])
restored_model.eval()
model.eval()
with torch.no_grad():
    sample_batch = test_data.tensors[0][:5].to(DEVICE)
    original_logits, original_boxes = model(sample_batch)
    restored_logits, restored_boxes = restored_model(sample_batch)
assert torch.allclose(original_logits, restored_logits)
assert torch.allclose(original_boxes, restored_boxes)
print("Reloaded model reproduces the predictions.")


## Recap and next step
You trained an actual network with shared visual features, classification and localization heads, validation checkpoint selection, held-out metrics, and a distribution-shift check. These are reusable habits; real detectors add more complex outputs and matching rules.

**Next:** [11 · Faster R-CNN and transfer learning](11_Faster_RCNN_and_Transfer_Learning.ipynb), or revisit [03 · IoU and NMS](03_IoU_and_Non_Maximum_Suppression.ipynb).

**References:** [PyTorch optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html), [saving model weights](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html).
